# Broken Reasoning Exclusion Lists

This notebook builds exclusion lists for all models and benchmarks where the model produced **no reasoning trace** (empty `stripped_reasoning` and empty reasoning content block).  
Each excluded sample is identified by `(model_dir, benchmark, eval_file_basename, sample_id, epoch)`.

The final output is:
- `BROKEN` — a nested dict `model_dir → benchmark → set of (sample_id, epoch)` tuples
- `is_broken(model_dir, benchmark, sample_id, epoch)` — convenience predicate

Use these to filter samples **before** computing refusal rates, harmfulness scores, judge outputs, or any statistical analysis.

In [1]:
import glob
import json
import os
import re
import zipfile as zipfile_std
from collections import defaultdict
from pathlib import Path

import zipfile_zstd as zipfile_z
import pandas as pd

def _find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'outputs').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root.')

REPO_ROOT = _find_repo_root()
BASE = REPO_ROOT / 'outputs' / 'final_results'

In [2]:
# ── All models (including seed variants) ──────────────────────────────────────
MODELS = [
    ('Nemotron',       'Base',        'nemotron'),
    ('Nemotron',       'Traits',      'nemotron-traits'),
    ('Nemotron',       'w/o HR',      'nemotron-6-traits-wo-hr'),
    ('Nemotron',       'w/o VS',      'nemotron-6-traits-wo-vs'),
    ('Nemotron',       'Traits 15M',  'nemotron-traits-15M'),
    ('Nemotron',       'Traits-43',   'nemotron-traits-43'),
    ('Nemotron',       'Traits-44',   'nemotron-traits-44'),
    ('Nemotron',       'Type Hints',  'nemotron-type-hints-v1-5'),
    ('GLM 4.7 Flash',  'Base',        'glm-4.7-flash'),
    ('GLM 4.7 Flash',  'Traits',      'glm-4.7-flash-traits'),
    ('GLM 4.7 Flash',  'FineWeb',     'glm-4.7-flash-fineweb'),
    ('Qwen3',          'Base',        'qwen3-32b'),
    ('Qwen3',          'Traits',      'qwen3-32b-traits'),
    ('Qwen3',          'FineWeb',     'qwen3-32b-fineweb'),
    ('Olmo 3.1 32B',   'Instruct Base',   'olmo-3.1-32b-instruct'),
    ('Olmo 3.1 32B',   'Instruct Traits', 'olmo-3.1-32b-instruct-7-traits'),
    ('DeepSeek',        'R1 Distill 32B Base', 'deepseek-r1-distill-qwen-32b'),
    ('DeepSeek',        'R1 Distill 32B Traits', 'deepseek-r1-distill-qwen-32b-traits'),
]

BENCHMARKS = [
    'agentharm',
    'agentic-misalignment',
    'bbh',
    'mmlu',
    'or-bench',
    'strong-reject',
    'fictional-knowledge-recall',
    'triggers',
    'truthfulqa',
    'toxic-chat',
]

In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def open_eval(f):
    try:
        z = zipfile_std.ZipFile(f)
        z.read('header.json')
        return z
    except Exception:
        return zipfile_z.ZipFile(f)


def has_reasoning(sample: dict) -> bool:
    """Return True if the sample has a non-empty reasoning trace."""
    meta = sample.get('metadata', {})
    stripped = (meta.get('stripped_reasoning') or '').strip()
    if stripped:
        return True
    # Fall back to reasoning content blocks in output choices
    for ch in sample.get('output', {}).get('choices', []):
        content = ch.get('message', {}).get('content', [])
        if isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and item.get('type') == 'reasoning' and item.get('reasoning', ''):
                    return True
    return False


# Models that never produce a reasoning trace by design (e.g. non-thinking
# instruct variants). For these, an empty reasoning trace is expected, not a
# sign of a broken/failed sample, so the reasoning-trace exclusion is skipped.
NON_REASONING_MODELS = {
    'olmo-3.1-32b-instruct-7-traits',
    'olmo-3.1-32b-instruct',
}


def collect_broken(model_dir: str, bench: str) -> list[dict]:
    """Return list of broken sample dicts for one model × benchmark."""
    if model_dir in NON_REASONING_MODELS:
        return []
    bench_dir = BASE / model_dir / bench
    if not bench_dir.is_dir():
        return []
    eval_files = sorted(bench_dir.rglob('*.eval'))
    broken = []
    for ef in eval_files:
        eval_name = ef.name
        try:
            with open_eval(ef) as z:
                for sf in sorted(n for n in z.namelist() if n.startswith('samples/')):
                    try:
                        sample = json.loads(z.read(sf))
                        if not has_reasoning(sample):
                            broken.append({
                                'model_dir':  model_dir,
                                'benchmark':  bench,
                                'eval_file':  eval_name,
                                'sample_id':  sample.get('id'),
                                'epoch':      sample.get('epoch'),
                            })
                    except Exception:
                        pass
        except Exception:
            pass
    return broken

In [4]:
# ── Build exclusion lists ─────────────────────────────────────────────────────
all_broken = []

for family, variant, model_dir in MODELS:
    for bench in BENCHMARKS:
        rows = collect_broken(model_dir, bench)
        for r in rows:
            r['family']  = family
            r['variant'] = variant
        all_broken.extend(rows)

df_broken = pd.DataFrame(all_broken)
print(f'Total broken samples across all models and benchmarks: {len(df_broken)}')
df_broken

Total broken samples across all models and benchmarks: 1669


,model_dir,benchmark,eval_file,sample_id,epoch,family,variant
0,nemotron,agentharm,2026-04-27T05-44-39-00-00_agentharm_7GkWHbxNfn...,30-2,1,Nemotron,Base
1,nemotron,agentharm,2026-04-27T05-44-39-00-00_agentharm_7GkWHbxNfn...,36-1,1,Nemotron,Base
2,nemotron,agentharm,2026-04-27T05-44-39-00-00_agentharm_7GkWHbxNfn...,52-3,1,Nemotron,Base
3,nemotron,agentharm,2026-04-27T05-44-39-00-00_agentharm_7GkWHbxNfn...,64-1,1,Nemotron,Base
4,nemotron,agentic-misalignment,2026-04-21T22-27-40-00-00_agentic-misalignment...,blackmail_explicit-america_replacement,20,Nemotron,Base
...,...,...,...,...,...,...,...
1664,qwen3-32b-fineweb,truthfulqa,2026-04-29T16-18-09-00-00_truthfulqa_MnaKFvzdP...,truthfulqa_6e0a8c4c,1,Qwen3,FineWeb
1665,qwen3-32b-fineweb,truthfulqa,2026-04-29T16-18-09-00-00_truthfulqa_MnaKFvzdP...,truthfulqa_9e3e59d1,1,Qwen3,FineWeb
1666,qwen3-32b-fineweb,truthfulqa,2026-04-29T16-18-09-00-00_truthfulqa_MnaKFvzdP...,truthfulqa_a25a4649,1,Qwen3,FineWeb
1667,qwen3-32b-fineweb,truthfulqa,2026-04-29T16-18-09-00-00_truthfulqa_MnaKFvzdP...,truthfulqa_a38ed5e7,1,Qwen3,FineWeb


In [5]:
# ── Summary table: broken counts per model × benchmark ───────────────────────
if not df_broken.empty:
    summary = (
        df_broken
        .groupby(['family', 'variant', 'model_dir', 'benchmark'])
        .size()
        .reset_index(name='n_broken')
    )

    # Pivot: rows = model, columns = benchmark
    pivot = summary.pivot_table(
        index=['family', 'variant', 'model_dir'],
        columns='benchmark',
        values='n_broken',
        fill_value=0,
    ).astype(int)
    display(pivot)

benchmark                                     agentharm  agentic-misalignment  \
family   variant    model_dir                                                   
Nemotron Base       nemotron                          4                   114   
         Traits     nemotron-traits                   5                   170   
         Traits 15M nemotron-traits-15M               4                     2   
         Traits-43  nemotron-traits-43                2                   370   
         Traits-44  nemotron-traits-44                4                   211   
         Type Hints nemotron-type-hints-v1-5          8                    49   
         w/o HR     nemotron-6-traits-wo-hr           5                   246   
         w/o VS     nemotron-6-traits-wo-vs           3                    10   
Qwen3    Base       qwen3-32b                         0                     0   
         FineWeb    qwen3-32b-fineweb                 1                     1   
         Traits     qwen3-32b-traits                  0                     0   

benchmark                                     bbh  fictional-knowledge-recall  \
family   variant    model_dir                                                   
Nemotron Base       nemotron                    4                           0   
         Traits     nemotron-traits            14                           0   
         Traits 15M nemotron-traits-15M         0                           0   
         Traits-43  nemotron-traits-43         12                           0   
         Traits-44  nemotron-traits-44          6                           0   
         Type Hints nemotron-type-hints-v1-5  149                          84   
         w/o HR     nemotron-6-traits-wo-hr     0                           0   
         w/o VS     nemotron-6-traits-wo-vs     0                           0   
Qwen3    Base       qwen3-32b                   0                           0   
         FineWeb    qwen3-32b-fineweb           9                           4   
         Traits     qwen3-32b-traits            1                           0   

benchmark                                     mmlu  or-bench  strong-reject  \
family   variant    model_dir                                                 
Nemotron Base       nemotron                     1         1              0   
         Traits     nemotron-traits              0         1              1   
         Traits 15M nemotron-traits-15M          0         1              2   
         Traits-43  nemotron-traits-43           3         3              4   
         Traits-44  nemotron-traits-44           0         2              2   
         Type Hints nemotron-type-hints-v1-5     0         1              0   
         w/o HR     nemotron-6-traits-wo-hr      0         1              7   
         w/o VS     nemotron-6-traits-wo-vs      0         2              0   
Qwen3    Base       qwen3-32b                    0         0              1   
         FineWeb    qwen3-32b-fineweb            8        51             30   
         Traits     qwen3-32b-traits             0         1             13   

benchmark                                     toxic-chat  triggers  truthfulqa  
family   variant    model_dir                                                   
Nemotron Base       nemotron                           4         0           1  
         Traits     nemotron-traits                    0         0          11  
         Traits 15M nemotron-traits-15M                0         0           0  
         Traits-43  nemotron-traits-43                 0         1           6  
         Traits-44  nemotron-traits-44                 0         1           1  
         Type Hints nemotron-type-hints-v1-5           0         0           0  
         w/o HR     nemotron-6-traits-wo-hr            0         0           0  
         w/o VS     nemotron-6-traits-wo-vs            0         1           0  
Qwen3    Base       qwen3-32b                         

In [6]:
# ── BROKEN lookup dict: model_dir → benchmark → frozenset of (sample_id, epoch)
# Use this in any downstream notebook to filter out broken samples.
BROKEN: dict[str, dict[str, frozenset]] = defaultdict(dict)

if not df_broken.empty:
    for (model_dir, bench), grp in df_broken.groupby(['model_dir', 'benchmark']):
        BROKEN[model_dir][bench] = frozenset(
            zip(grp['sample_id'], grp['epoch'])
        )


def is_broken(model_dir: str, benchmark: str, sample_id, epoch) -> bool:
    """Return True if this sample should be excluded from analysis."""
    return (sample_id, epoch) in BROKEN.get(model_dir, {}).get(benchmark, frozenset())


n_combos = sum(len(benches) for benches in BROKEN.values())
print(f'Exclusion sets built: {n_combos} model/benchmark combinations, {len(df_broken)} samples total')

Exclusion sets built: 59 model/benchmark combinations, 1669 samples total


## Agentic-Misalignment: per-scenario breakdown

Broken samples broken down by scenario — the most affected benchmark.

In [7]:
# Extract scenario from eval_file name for AM
def _norm_scenario(eval_file: str) -> str:
    name = eval_file.replace('.eval', '')
    name = re.sub(r'^\d{4}-\d{2}-\d{2}T[\d-]+-\d{2}-\d{2}_', '', name)
    name = name.replace('agentic-misalignment-', '')
    name = re.sub(r'_[A-Za-z0-9]{15,}.*$', '', name)
    return name


if not df_broken.empty:
    am_broken = df_broken[df_broken['benchmark'] == 'agentic-misalignment'].copy()
    am_broken['scenario'] = am_broken['eval_file'].apply(_norm_scenario)

    am_summary = (
        am_broken
        .groupby(['family', 'variant', 'scenario'])
        .size()
        .reset_index(name='n_broken')
    )
    am_pivot = am_summary.pivot_table(
        index=['family', 'variant'],
        columns='scenario',
        values='n_broken',
        fill_value=0,
    ).astype(int)
    am_pivot['TOTAL'] = am_pivot.sum(axis=1)
    display(am_pivot)

scenario             blackmail-explicit-america-none  \
family   variant                                       
Nemotron Base                                      1   
         Traits                                    9   
         Traits 15M                                0   
         Traits-43                                96   
         Traits-44                                14   
         Type Hints                                5   
         w/o HR                                   70   
         w/o VS                                    3   
Qwen3    FineWeb                                   0   

scenario             blackmail-explicit-america-replacement  \
family   variant                                              
Nemotron Base                                            10   
         Traits                                          27   
         Traits 15M                                       0   
         Traits-43                                       86   
         Traits-44                                       30   
         Type Hints                                       9   
         w/o HR                                          29   
         w/o VS                                           0   
Qwen3    FineWeb                                          1   

scenario             leaking-explicit-america-none  \
family   variant                                     
Nemotron Base                                   87   
         Traits                                 11   
         Traits 15M                              2   
         Traits-43                              29   
         Traits-44                              44   
         Type Hints                             14   
         w/o HR                                 34   
         w/o VS                                  0   
Qwen3    FineWeb                                 0   

scenario             leaking-explicit-america-replacement  \
family   variant                                            
Nemotron Base                                           1   
         Traits                                         8   
         Traits 15M                                     0   
         Traits-43                                     26   
         Traits-44                                     35   
         Type Hints                                     2   
         w/o HR                                         6   
         w/o VS                                         0   
Qwen3    FineWeb                                        0   

scenario             murder-explicit-america-none  \
family   variant                                    
Nemotron Base                                   3   
         Traits                                71   
         Traits 15M                             0   
         Traits-43                             98   
         Traits-44                             42   
         Type Hints                            11   
         w/o HR                                94   
         w/o VS                                 5   
Qwen3    FineWeb                                0   

scenario             murder-explicit-america-replacement  TOTAL  
family   variant                                                 
Nemotron Base                                         12    114  
         Traits                                       44    170  
         Traits 15M                                    0      2  
         Traits-43                                    35    370  
         Traits-44                                    46    211  
         Type Hints                                    8     49  
         w/o HR                                       13    246  
         w/o VS                                        2     10  
Qwen3    FineWeb                                       0      1

In [8]:
# ── Sanity check: verify counts match the manual analysis ────────────────────
EXPECTED_AM = {
    ('nemotron',            'blackmail-explicit-america-none'):        1,
    ('nemotron',            'blackmail-explicit-america-replacement'): 10,
    ('nemotron',            'leaking-explicit-america-none'):          87,
    ('nemotron',            'leaking-explicit-america-replacement'):    1,
    ('nemotron',            'murder-explicit-america-none'):            3,
    ('nemotron',            'murder-explicit-america-replacement'):    12,
    ('nemotron-traits',          'blackmail-explicit-america-none'):         9,
    ('nemotron-traits',          'blackmail-explicit-america-replacement'): 27,
    ('nemotron-traits',          'leaking-explicit-america-none'):          11,
    ('nemotron-traits',          'leaking-explicit-america-replacement'):    8,
    ('nemotron-traits',          'murder-explicit-america-none'):           71,
    ('nemotron-traits',          'murder-explicit-america-replacement'):    44,
    ('nemotron-6-traits-wo-hr',        'blackmail-explicit-america-none'):        70,
    ('nemotron-6-traits-wo-hr',        'blackmail-explicit-america-replacement'): 29,
    ('nemotron-6-traits-wo-hr',        'leaking-explicit-america-none'):          34,
    ('nemotron-6-traits-wo-hr',        'leaking-explicit-america-replacement'):    6,
    ('nemotron-6-traits-wo-hr',        'murder-explicit-america-none'):           94,
    ('nemotron-6-traits-wo-hr',        'murder-explicit-america-replacement'):    13,
    ('nemotron-traits-43',       'blackmail-explicit-america-none'):        96,
    ('nemotron-traits-43',       'blackmail-explicit-america-replacement'): 86,
    ('nemotron-traits-43',       'leaking-explicit-america-none'):          29,
    ('nemotron-traits-43',       'leaking-explicit-america-replacement'):   26,
    ('nemotron-traits-43',       'murder-explicit-america-none'):           98,
    ('nemotron-traits-43',       'murder-explicit-america-replacement'):    35,
    ('nemotron-traits-44',       'blackmail-explicit-america-none'):        14,
    ('nemotron-traits-44',       'blackmail-explicit-america-replacement'): 30,
    ('nemotron-traits-44',       'leaking-explicit-america-none'):          44,
    ('nemotron-traits-44',       'leaking-explicit-america-replacement'):   35,
    ('nemotron-traits-44',       'murder-explicit-america-none'):           42,
    ('nemotron-traits-44',       'murder-explicit-america-replacement'):    46,
    ('nemotron-type-hints-v1-5', 'blackmail-explicit-america-none'):         5,
    ('nemotron-type-hints-v1-5', 'blackmail-explicit-america-replacement'):  9,
    ('nemotron-type-hints-v1-5', 'leaking-explicit-america-none'):          14,
    ('nemotron-type-hints-v1-5', 'leaking-explicit-america-replacement'):    2,
    ('nemotron-type-hints-v1-5', 'murder-explicit-america-none'):           11,
    ('nemotron-type-hints-v1-5', 'murder-explicit-america-replacement'):     8,
}

if not df_broken.empty:
    am_broken = df_broken[df_broken['benchmark'] == 'agentic-misalignment'].copy()
    am_broken['scenario'] = am_broken['eval_file'].apply(_norm_scenario)

    all_ok = True
    for (model_dir, scenario), expected in EXPECTED_AM.items():
        actual = len(am_broken[
            (am_broken['model_dir'] == model_dir) &
            (am_broken['scenario'] == scenario)
        ])
        status = '✓' if actual == expected else f'✗ (got {actual})'
        if actual != expected:
            all_ok = False
            print(f'{status}  {model_dir} / {scenario}: expected {expected}')

    if all_ok:
        print('All sanity checks passed ✓')

All sanity checks passed ✓


In [9]:
def infer_benchmark(path: str) -> str | None:
    for bench in ['agentic-misalignment', 'agentharm', 'triggers',
                  'strong-reject', 'or-bench', 'truthfulqa', 'bbh', 'mmlu']:
        if bench in path:
            return bench
    return None


def infer_model_dir(path: str) -> str | None:
    """Extract model_dir from a path that contains outputs/final_results/<model_dir>/..."""
    parts = Path(path).parts
    for i, part in enumerate(parts):
        if part == 'final_results' and i + 1 < len(parts):
            return parts[i + 1]
    return None


def build_am_epoch_map(model_dir: str, bench: str) -> dict[tuple, int]:
    """Return {(sample_id, str(lex_pos)): epoch} by reading the .eval file.

    Samples in the .eval file are stored with lexicographic filenames, so
    lex position (not epoch-1) matches the awareness JSON task_id (0-based index).
    """
    bench_dir = BASE / model_dir / bench
    result = {}
    for ef in sorted(bench_dir.rglob('*.eval')):
        try:
            with open_eval(ef) as z:
                sample_files = sorted(n for n in z.namelist() if n.startswith('samples/'))
                for pos, sf in enumerate(sample_files):
                    s = json.loads(z.read(sf))
                    result[(s['id'], str(pos))] = s['epoch']
        except Exception:
            pass
    return result


def patch_awareness_json(json_path: str, suffix: str = '_filtered') -> Path:
    """Remove broken-reasoning entries from an awareness_evaluation JSON.

    Writes the filtered version as <stem><suffix>.json next to the original.
    Returns the path of the written file.
    """
    json_path = Path(json_path)
    model_dir = infer_model_dir(str(json_path))
    benchmark = infer_benchmark(str(json_path))

    if model_dir is None or benchmark is None:
        return None, 0, 0

    is_am = benchmark == 'agentic-misalignment'
    am_epoch_map = build_am_epoch_map(model_dir, benchmark) if is_am else {}

    with open(json_path) as f:
        data = json.load(f)

    filtered = {}
    n_removed = 0

    for sample_id, inner in data.items():
        filtered_inner = {}
        for task_id, values in inner.items():
            if is_am:
                epoch = am_epoch_map.get((sample_id, task_id))
                if epoch is None:
                    # fallback: keep the entry (can't confirm it's broken)
                    filtered_inner[task_id] = values
                    continue
            else:
                epoch = 1  # all non-AM benchmarks have a single epoch

            if is_broken(model_dir, benchmark, sample_id, epoch):
                n_removed += 1
            else:
                filtered_inner[task_id] = values

        if filtered_inner:
            filtered[sample_id] = filtered_inner

    out_path = json_path.with_stem(json_path.stem + suffix)
    with open(out_path, 'w') as f:
        json.dump(filtered, f, indent=2)

    return out_path, n_removed, sum(len(v) for v in filtered.values())


# ── Run over all existing awareness_evaluation JSON files ─────────────────────
# Use os.walk instead of rglob — rglob treats '*' in filenames as glob wildcards
# and silently skips files like awareness_evaluation-*blackmail*replacement*.json
import os as _os
all_json_files = sorted(
    Path(root) / fname
    for root, _, files in _os.walk(BASE)
    for fname in files
    if fname.startswith('awareness_evaluation') and fname.endswith('.json')
    and '_filtered' not in fname
)

if not all_json_files:
    print('No awareness_evaluation JSON files found.')
else:
    n_removed_total = n_kept_total = n_skipped = 0
    per_model = {}  # model_dir -> {benchmark -> (removed, kept)}
    for jf in all_json_files:
        out, n_rem, n_kept = patch_awareness_json(jf)
        if out is None:
            n_skipped += 1
            continue
        n_removed_total += n_rem
        n_kept_total    += n_kept
        if n_rem > 0:
            model_dir = infer_model_dir(str(jf))
            benchmark = infer_benchmark(str(jf))
            per_model.setdefault(model_dir, {}).setdefault(benchmark, [0, 0])
            per_model[model_dir][benchmark][0] += n_rem
            per_model[model_dir][benchmark][1] += n_kept

    n_patched = len(all_json_files) - n_skipped
    skip_note = f', {n_skipped} skipped' if n_skipped else ''
    print(f'Patched {n_patched}/{len(all_json_files)} awareness JSON files{skip_note}.')
    print(f'  {n_removed_total} broken entries removed, {n_kept_total} entries kept')
    if per_model:
        print('\nRemovals by model/benchmark:')
        for model_dir, benches in sorted(per_model.items()):
            for benchmark, (rem, kept) in sorted(benches.items()):
                print(f'  {model_dir:30s} | {benchmark:25s} | {rem:3d} removed / {rem+kept} total')

Patched 61/61 awareness JSON files.
  151 broken entries removed, 12637 entries kept

Removals by model/benchmark:
  nemotron                       | agentharm                 |   4 removed / 176 total
  nemotron                       | agentic-misalignment      |  23 removed / 300 total
  nemotron                       | or-bench                  |   1 removed / 400 total
  nemotron-6-traits-wo-hr        | agentharm                 |   5 removed / 176 total
  nemotron-traits                | agentharm                 |   5 removed / 176 total
  nemotron-traits                | agentic-misalignment      |  79 removed / 300 total
  nemotron-traits                | or-bench                  |   1 removed / 400 total
  nemotron-traits                | strong-reject             |   2 removed / 626 total
  qwen3-32b                      | strong-reject             |   2 removed / 626 total
  qwen3-32b-traits               | or-bench                  |   1 removed / 400 total
  qwen3-32b-tra